# End-to-End Pipeline Demo

Runs the full agent-based model sequence on ~10 agents:
synthesize → persona → work/school zone → activities → schedule →
destinations → validate → tours → mode choice.

**Requires** an `ANTHROPIC_API_KEY` environment variable.

## 1. Imports

In [ ]:
from collections import Counter

from aibm import ZoneSpec, synthesize_population
from aibm.agent import ModeOption
from aibm.day_plan import DayPlan
from aibm.llm import AnthropicClient
from aibm.zone import Zone

## 2. Configuration

Set all model inputs here before running the rest of the notebook:

- `AGENT_MODEL` — which LLM to use for each agent
- `MAX_TOKENS` — maximum tokens the LLM may return per call
- `POPULATION_SEED` — random seed for reproducible population synthesis
- `OVERWRITE` — set `True` to re-generate personas / zone choices on re-runs
- `zones` — spatial units in the model area
- `travel_times` — minutes per mode for every origin–destination pair
- `zone_specs` — household and demographic parameters per zone

In [ ]:
# ── LLM settings ───────────────────────────────────────────────────────────────
AGENT_MODEL = "claude-haiku-4-5-20251001"
MAX_TOKENS = 1024
POPULATION_SEED = 42

# ── Re-run behaviour ────────────────────────────────────────────────────────────
# Set True to let the LLM overwrite already-generated personas / zone choices
# when a cell is re-executed.
OVERWRITE = False

# ── Zones ───────────────────────────────────────────────────────────────────────
zones = [
    Zone(
        id="north",
        name="North Suburb",
        x=0.0,
        y=1.0,
        land_use={"residential": True, "commercial": False, "school": False},
    ),
    Zone(
        id="centre",
        name="City Centre",
        x=0.0,
        y=0.0,
        land_use={"residential": False, "commercial": True, "school": False},
    ),
    Zone(
        id="east",
        name="East Side",
        x=1.0,
        y=0.0,
        land_use={"residential": True, "commercial": True, "school": True},
    ),
]

# ── Travel times (minutes) per mode ────────────────────────────────────────────
travel_times = {
    ("north", "north"): {"car": 5, "transit": 10, "bike": 8, "walk": 15},
    ("north", "centre"): {"car": 15, "transit": 25, "bike": 30, "walk": 60},
    ("north", "east"): {"car": 20, "transit": 35, "bike": 40, "walk": 80},
    ("centre", "north"): {"car": 15, "transit": 25, "bike": 30, "walk": 60},
    ("centre", "centre"): {"car": 5, "transit": 8, "bike": 7, "walk": 12},
    ("centre", "east"): {"car": 10, "transit": 20, "bike": 25, "walk": 50},
    ("east", "north"): {"car": 20, "transit": 35, "bike": 40, "walk": 80},
    ("east", "centre"): {"car": 10, "transit": 20, "bike": 25, "walk": 50},
    ("east", "east"): {"car": 5, "transit": 10, "bike": 8, "walk": 15},
}

# ── Population zone specs ───────────────────────────────────────────────────────
zone_specs = [
    ZoneSpec(
        zone_id="north",
        n_households=2,
        household_size_dist={1: 0.2, 2: 0.5, 3: 0.3},
        age_dist={"0-17": 0.15, "18-64": 0.70, "65+": 0.15},
        employment_rate=0.70,
        student_rate=0.10,
        vehicle_dist={0: 0.1, 1: 0.6, 2: 0.3},
        license_rate=0.85,
    ),
    ZoneSpec(
        zone_id="east",
        n_households=2,
        household_size_dist={1: 0.3, 2: 0.4, 3: 0.3},
        age_dist={"0-17": 0.10, "18-64": 0.75, "65+": 0.15},
        employment_rate=0.50,
        student_rate=0.25,
        vehicle_dist={0: 0.4, 1: 0.45, 2: 0.15},
        license_rate=0.65,
    ),
]

In [ ]:
client = AnthropicClient(max_tokens=MAX_TOKENS)
print("Client ready.")

## 3. Zone lookup

In [ ]:
zone_lookup = {z.id: z for z in zones}
print(f"{len(zones)} zones defined.")

## 4. Travel-time helper

In [ ]:
def get_travel_times_from(home: str) -> dict[str, dict[str, float]]:
    """Get travel times from a home zone to all other zones."""
    return {
        dest: modes
        for (orig, dest), modes in travel_times.items()
        if orig == home
    }


print("Travel times defined for all zone pairs.")

## 5. Synthesize ~10 agents

In [ ]:
households = synthesize_population(zone_specs, seed=POPULATION_SEED)
all_agents = [agent for hh in households for agent in hh.members]

for agent in all_agents:
    agent.model = AGENT_MODEL

print(f"Households: {len(households)}, Agents: {len(all_agents)}")
for hh in households:
    print(f"  {hh}  zone={hh.home_zone}  vehicles={hh.num_vehicles}")
    for a in hh.members:
        lic = "licence" if a.has_license else "no licence"
        print(f"    {a.name}  age={a.age}  {a.employment}  {lic}")

## 6. Generate personas

In [ ]:
hh_lookup = {a.id: hh for hh in households for a in hh.members}

for agent in all_agents:
    hh = hh_lookup[agent.id]
    agent.generate_persona(client=client, household=hh, overwrite=OVERWRITE)
    print(f"{agent.name}: {agent.persona}")

## 7. Choose work/school zones

In [ ]:
for agent in all_agents:
    home = agent.home_zone
    assert home is not None
    tt = get_travel_times_from(home)

    if agent.employment == "employed":
        zone_id = agent.choose_work_zone(
            zones, tt, client=client, overwrite=OVERWRITE
        )
        print(f"{agent.name} works in: {zone_id}")
    elif agent.employment == "student":
        zone_id = agent.choose_school_zone(
            zones, tt, client=client, overwrite=OVERWRITE
        )
        print(f"{agent.name} studies in: {zone_id}")
    else:
        print(f"{agent.name}: no work/school zone needed ({agent.employment})")

## 8. Generate activities

In [15]:
agent_activities: dict[str, list] = {}

for agent in all_agents:
    activities = agent.generate_activities(client=client)
    agent_activities[agent.id] = activities
    types = [a.type for a in activities]
    print(f"{agent.name}: {types}")

Agent 0: ['work', 'grocery shopping', 'lunch', 'commute home', 'leisure/relaxation']
Agent 1: ['school', 'homework', 'leisure - cycling with friends', 'shopping - groceries or supplies', 'community center activity']
Agent 2: ['work', 'grocery shopping', 'meal preparation', 'leisure/relaxation']
Agent 3: ['School', 'Library study session', 'Grocery shopping', 'Coffee with friends', 'Lunch']
Agent 4: ['work', 'grocery shopping', 'lunch', 'household chores', 'leisure/relaxation']
Agent 5: ['work', 'grocery shopping', 'commute via public transportation', 'meal preparation', 'leisure/relaxation']
Agent 6: ['job search', 'social service appointment', 'grocery shopping', 'leisure - visit local park']
Agent 7: ['Medical appointment', 'Grocery shopping', 'Household errands', 'Social visit']
Agent 8: ['school', 'lunch at home', 'park play', 'snack time', 'family errands with parent', 'dinner with family']
Agent 9: ['breakfast', 'shopping', 'lunch', 'social visit', 'cultural event', 'dining']
Age

## 9. Schedule activities

In [ ]:
agent_plans: dict[str, DayPlan] = {}

for agent in all_agents:
    activities = agent_activities[agent.id]
    day_plan = agent.schedule_activities(activities, client=client)
    agent_plans[agent.id] = day_plan
    print(f"\n{agent.name}'s schedule:")
    for act in day_plan.activities:
        start = f"{act.start_time:6.0f}" if act.start_time is not None else "   N/A"
        end = f"{act.end_time:6.0f}" if act.end_time is not None else "   N/A"
        print(f"  {act.type:20s}  {start}-{end}")

## 10. Choose destinations

In [ ]:
for agent in all_agents:
    for activity in agent_activities[agent.id]:
        if activity.location is None:
            agent.choose_destination(activity, candidates=zones, client=client)
            print(f"{agent.name} → {activity.type} at {activity.location}")

## 11. Validate day plans

In [ ]:
for agent in all_agents:
    day_plan = agent_plans[agent.id]
    warnings = day_plan.validate()
    if warnings:
        print(f"{agent.name}: {len(warnings)} warning(s)")
        for w in warnings:
            print(f"  ⚠ {w}")
    else:
        print(f"{agent.name}: OK")

## 12. Build tours

In [ ]:
for agent in all_agents:
    day_plan = agent_plans[agent.id]
    agent.build_tours(day_plan)
    n_tours = len(day_plan.tours)
    n_trips = len(day_plan.trips)
    print(f"\n{agent.name}: {n_tours} tour(s), {n_trips} trip(s)")
    for i, tour in enumerate(day_plan.tours):
        print(f"  Tour {i + 1} (closed={tour.is_closed}):")
        for trip in tour.trips:
            print(f"    {trip.origin} → {trip.destination}")

## 13. Choose mode per tour

In [ ]:
all_mode_choices = []

for agent in all_agents:
    hh = hh_lookup[agent.id]
    day_plan = agent_plans[agent.id]

    for tour in day_plan.tours:
        # Use outbound trip for travel time lookup
        first_trip = tour.trips[0]
        pair = (first_trip.origin, first_trip.destination)
        tt = travel_times.get(pair, {"walk": 30})

        # Build mode options, filtering car if no licence or 0 vehicles
        options = []
        for mode, minutes in tt.items():
            if mode == "car" and (not agent.has_license or hh.num_vehicles == 0):
                continue
            options.append(ModeOption(mode=mode, travel_time=minutes))

        if not options:
            options = [ModeOption(mode="walk", travel_time=30)]

        choice = agent.choose_tour_mode(tour, options, client=client, household=hh)
        all_mode_choices.append(choice)
        stops = " → ".join(t.destination for t in tour.trips)
        print(f"{agent.name}: {stops} by {choice.option.mode}")

## 14. Summary — mode share

In [ ]:
all_trips = [trip for agent in all_agents for trip in agent_plans[agent.id].trips]

mode_counts = Counter(t.mode for t in all_trips)
total = len(all_trips)

print(f"Total trips: {total}\n")
print("Mode share:")
for mode, count in mode_counts.most_common():
    print(f"  {mode:8s}: {count:3d}  ({count / total:.0%})")